# 04 · Binary packing (`pack_field`, `pack_frames`)

The HTML exporters embed data as **gzip-compressed little-endian float32, base64-encoded** instead of JSON.
You can call the packers directly to feed terraplot's `unpackField` / `unpackFrames` in your own app.

**Encoding pipeline:** `numpy float32 → little-endian → gzip → base64 ASCII`

**Single-field format (`TPLD`):** magic, version, nlon, nlat, `lons`, `lats`, `field` (row-major), JSON meta.
**Multi-frame format (`TPLF`):** adds `n_frames`, length-prefixed coord strings, and frame-major fields.

In [ ]:
import numpy as np
import xarray as xr
import pyterraplot  # registers the .tp accessor on DataArray and Dataset

def make_field(nlat=73, nlon=144, phase=0.0, name="t2m",
               long_name="2m temperature anomaly", units="K", holes=True):
    """A smooth, globe-shaped synthetic field on a regular lat/lon grid."""
    lats = np.linspace(90, -90, nlat)
    lons = np.linspace(-180, 180, nlon)
    LON, LAT = np.meshgrid(lons, lats)
    data = (
        8 * np.cos(np.radians(LAT)) * np.sin(np.radians(2 * LON) + phase)
        + 5 * np.sin(np.radians(3 * LON)) * np.cos(np.radians(2 * LAT))
        + 3 * np.cos(np.radians(5 * LON)) * np.sin(np.radians(LAT))
    ).astype(np.float32)
    if holes:
        rng = np.random.default_rng(0)
        data[rng.random((nlat, nlon)) < 0.02] = np.nan  # NaN "missing" cells
    return xr.DataArray(
        data, dims=["lat", "lon"], coords={"lat": lats, "lon": lons},
        name=name, attrs={"units": units, "long_name": long_name},
    )

da = make_field()
da

## `pack_field` — one field

In [ ]:
from pyterraplot import pack_field
import json

payload = da.tp.to_dict()
b64 = pack_field(payload)

json_bytes = len(json.dumps(payload).encode())
bin_bytes  = len(b64.encode())
print(f"JSON   payload : {json_bytes/1024:7.1f} kB")
print(f"packed base64  : {bin_bytes/1024:7.1f} kB  ({json_bytes/bin_bytes:.1f}× smaller)")
print("base64 head    :", b64[:48], "...")

## Peek inside the TPLD header

We decode the base64 + gzip ourselves to confirm the magic and shape.

In [ ]:
import base64, gzip, struct
raw = gzip.decompress(base64.b64decode(b64))
magic, version, nlon, nlat = struct.unpack("<IIII", raw[:16])
print("magic  :", magic.to_bytes(4, "little").decode(), "(expect TPLD)")
print("version:", version)
print("nlon   :", nlon, " nlat:", nlat, " -> matches", len(payload["lons"]), len(payload["lats"]))

# trailing JSON metadata
meta_len = struct.unpack("<I", raw[-4 - 0:][:4])[0] if False else None

## `pack_frames` — an animation

Takes a `frames_compact` dict and packs every frame into one buffer (magic `TPLF`).

In [ ]:
from pyterraplot import pack_frames

cube = xr.concat(
    [make_field(phase=p, holes=False) for p in np.linspace(0, 2*np.pi, 8, endpoint=False)],
    dim="time",
).assign_coords(time=np.arange(8))
cube.name, cube.attrs = "t2m", {"units": "K", "long_name": "anim"}

compact = cube.tp.frames_compact(dim="time")
fb64 = pack_frames(compact)

raw = gzip.decompress(base64.b64decode(fb64))
magic, version, nlon, nlat, n_frames = struct.unpack("<IIIII", raw[:20])
print("magic   :", magic.to_bytes(4, "little").decode(), "(expect TPLF)")
print("n_frames:", n_frames, " nlon:", nlon, " nlat:", nlat)
print("packed  :", f"{len(fb64)/1024:.0f} kB for {n_frames} frames")